# 03 – Combined EDA & Statistical Testing

**Research question:** How does the time players spend in a Steam game relate to their likelihood of recommending it, how does that differ across genres and gameplay styles, and what do long-time negative reviewers look like?

This notebook combines the two cleaned datasets:

| Dataset | Owner | File |
|---|---|---|
| Cleaned game data (Steam Store + SteamSpy) | Timothy – `01_game_cleaning.ipynb` | `data/cleaned/games_data.csv`, `data/cleaned/steamspy.json` |
| Cleaned individual reviews | D'Angelo – `02_review_cleaning.ipynb` | `data/individual_reviews_cleaned.csv` |
| Game-level review summary (for a sanity check) | Timothy | `data/cleaned/steam_reviews_summary.csv` |

Sections:
1. Load data
2. Build game-level features (primary genre, gameplay style, story-rich, free-to-play)
3. Merge on `appid` and verify the merge
4. EDA – playtime vs. recommendation
5. Genres and gameplay styles
6. Long-time (100+ hour) negative reviewers
7. Statistical tests
8. Summary of findings

> Charts are intentionally left out of this notebook; every result is a table so the visualizations can be built from the saved merged dataset.

In [1]:
import ast
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

DATA = "../data"
GAMES_FILE    = f"{DATA}/cleaned/games_data.csv"
STEAMSPY_FILE = f"{DATA}/cleaned/steamspy.json"      # saved as CSV text despite the .json extension
SUMMARY_FILE  = f"{DATA}/cleaned/steam_reviews_summary.csv"
REVIEWS_FILE  = f"{DATA}/individual_reviews_cleaned.csv"
RAW_GAMES     = f"{DATA}/raw/games_data.csv"
MERGED_OUT    = f"{DATA}/cleaned/reviews_with_games.csv"

PLAYTIME_ORDER = ["0–2 hours", "2–10 hours", "10–50 hours", "50–100 hours", "100+ hours"]

## 1. Load the cleaned datasets

In [2]:
games    = pd.read_csv(GAMES_FILE)
steamspy = pd.read_csv(STEAMSPY_FILE)
summary  = pd.read_csv(SUMMARY_FILE)
reviews  = pd.read_csv(REVIEWS_FILE)

for name, df in [("games", games), ("steamspy", steamspy), ("summary", summary), ("reviews", reviews)]:
    print(f"{name:9s} rows={len(df):>5}  cols={df.shape[1]:>3}  duplicate appid/review ids="
          f"{(df['recommendationid'] if name == 'reviews' else df['appid']).duplicated().sum()}")

games     rows=  174  cols= 12  duplicate appid/review ids=0
steamspy  rows=  195  cols= 13  duplicate appid/review ids=0
summary   rows=  200  cols=  7  duplicate appid/review ids=0
reviews   rows= 6999  cols= 18  duplicate appid/review ids=0


In [3]:
print("Reviews – missing values:")
print(reviews.isna().sum()[lambda s: s > 0])
print("\nReviews per game:")
reviews.groupby("appid").size().describe()

Reviews – missing values:
playtime_at_review_min       1
review_text                 23
playtime_at_review_hours     1
playtime_group               1
dtype: int64

Reviews per game:


count     28.000000
mean     249.964286
std       93.873233
min        7.000000
25%      254.500000
50%      300.000000
75%      300.000000
max      300.000000
dtype: float64

The review file has one row with a missing `playtime_at_review` (so no playtime group) and 23 empty review texts. Review text is not used in this analysis, so only the row with missing playtime is dropped. Everything else from D'Angelo's cleaning (hours conversion, 1/0 recommendation, playtime groups) is used as-is.

We use **`playtime_at_review_hours`** rather than `playtime_forever_hours`, because it reflects how much the player had played *when they made the recommendation*.

In [4]:
rows_before = len(reviews)
reviews = reviews.dropna(subset=["playtime_at_review_hours", "playtime_group"]).copy()
reviews["playtime_group"] = pd.Categorical(reviews["playtime_group"], categories=PLAYTIME_ORDER, ordered=True)
print(f"Reviews: {rows_before} -> {len(reviews)} rows")

Reviews: 6999 -> 6998 rows


## 2. Game-level features

Steam Store genre lists are alphabetical, so "first genre in the list" is not a meaningful primary genre (most games would become *Action* or *Adventure*). Timothy's cleaning already produced `tag_main` from the user-voted SteamSpy tags mapped into broad genre groups, so that is used as the **primary genre**.

| Feature | Source | Rule |
|---|---|---|
| `primary_genre` | SteamSpy `tag_main` | Highest-voted user tag mapped to a genre group (Timothy) |
| `gameplay_style` | Store categories | `Multiplayer` if the game lists any multi-player category, else `Single-player only` |
| `story_rich` | SteamSpy tags | `True` if the user tags include *Story Rich* |
| `free_to_play` | Store `is_free` | as provided |

In [5]:
def has_tag(tag_str, tag):
    try:
        return tag in ast.literal_eval(tag_str)
    except (ValueError, SyntaxError):
        return False

spy = steamspy[["appid", "tag_main", "tags", "owners_mean"]].copy()
spy["story_rich"] = spy["tags"].apply(lambda t: has_tag(t, "Story Rich"))
spy = spy.drop(columns="tags").rename(columns={"tag_main": "primary_genre"})

game_cols = ["appid", "name", "genres", "singleplayer", "multiplayer", "is_free", "final_price", "release_date"]
game_features = games[game_cols].merge(spy, on="appid", how="outer", indicator="game_source")
game_features["game_source"].value_counts()

game_source
both          174
right_only     21
left_only       0
Name: count, dtype: int64

### Fixing the one review game missing from the cleaned game data

The reviews cover 28 games. One of them, **Command & Conquer: Red Alert 3 (appid 17480)**, was removed in Timothy's cleaning (it is no longer sold on Steam, so its price is missing) but it is still in the cleaned SteamSpy data. Dropping it would throw away its 300 reviews, and price is not needed here, so its gameplay flags are filled in from the raw Steam Store categories instead.

In [6]:
review_ids = set(reviews["appid"])
missing = review_ids - set(games["appid"])
print("Review games not in cleaned games_data:", missing)

raw = pd.read_csv(RAW_GAMES)
for appid in missing:
    raw_row = raw.loc[raw["appid"] == appid].iloc[0]
    cats = str(raw_row["categories"])
    idx = game_features["appid"] == appid
    game_features.loc[idx, "name"] = raw_row["name"]
    game_features.loc[idx, "singleplayer"] = "Single-player" in cats
    game_features.loc[idx, "multiplayer"] = "Multi-player" in cats
    game_features.loc[idx, "is_free"] = False
    print(raw_row["name"], "| categories:", cats)

Review games not in cleaned games_data: {17480}
Command & Conquer™ Red Alert™ 3 | categories: Single-player; Steam Workshop; Steam Cloud; Includes level editor; Family Sharing


In [7]:
# games that exist only in SteamSpy (not reviewed) stay NaN here
game_features["gameplay_style"] = game_features["multiplayer"].map({True: "Multiplayer", False: "Single-player only"})
game_features["free_to_play"] = game_features["is_free"].map({True: True, False: False, "True": True, "False": False})

review_games = game_features[game_features["appid"].isin(review_ids)]
print("Review games with features:", len(review_games), "of", len(review_ids))
review_games[["appid", "name", "primary_genre", "gameplay_style", "story_rich", "free_to_play"]].sort_values("primary_genre")

Review games with features: 28 of 28


,appid,name,primary_genre,gameplay_style,story_rich,free_to_play
141,1621140,Clawfish,Action,Single-player only,False,False
99,887720,Devil's Hunt,Action,Single-player only,True,False
44,356190,Middle-earth™: Shadow of War™,Action,Single-player only,True,False
114,1084570,The Lar,Adventure,Single-player only,False,True
166,2196980,Arcadie: Second-Born,Adventure,Single-player only,True,False
191,3364770,Misty Judgment,Adventure,Single-player only,True,False
190,3253440,EGG RAIDERS,Horror,Multiplayer,False,False
124,1304930,The Outlast Trials,Horror,Multiplayer,False,False
160,1929610,Demonologist,Horror,Multiplayer,False,False
129,1428820,Wordle 3,Puzzle,Single-player only,False,False


## 3. Merge reviews with game features on `appid`

In [8]:
features_for_merge = game_features.loc[game_features["appid"].isin(review_ids),
    ["appid", "name", "primary_genre", "gameplay_style", "story_rich", "free_to_play", "owners_mean", "final_price"]]

merged = reviews.merge(features_for_merge, on="appid", how="left",
                       validate="many_to_one", indicator=True)
merged["_merge"].value_counts()

_merge
both          6998
left_only        0
right_only       0
Name: count, dtype: int64

### Merge verification

In [9]:
checks = {
    "rows before merge (reviews)": len(reviews),
    "rows after merge": len(merged),
    "row count unchanged": len(merged) == len(reviews),
    "duplicate review ids after merge": merged["recommendationid"].duplicated().sum(),
    "reviews with no matching game": (merged["_merge"] != "both").sum(),
    "reviews missing primary_genre": merged["primary_genre"].isna().sum(),
    "reviews missing gameplay_style": merged["gameplay_style"].isna().sum(),
    "unique games after merge": merged["appid"].nunique(),
}
assert checks["row count unchanged"] and checks["duplicate review ids after merge"] == 0
assert checks["reviews with no matching game"] == 0
pd.Series(checks, name="value").to_frame()

,value
rows before merge (reviews),6998
rows after merge,6998
row count unchanged,True
duplicate review ids after merge,0
reviews with no matching game,0
reviews missing primary_genre,0
reviews missing gameplay_style,0
unique games after merge,28


**Sample sanity check:** each game's positive rate in our review sample should be close to its overall Steam positive percentage from the game-level summary. Large gaps would mean our 300-review sample is unrepresentative for that game (recent reviews are weighted toward the newest players).

In [10]:
sample_rate = merged.groupby("appid")["recommendation"].agg(sample_reviews="size", sample_positive_pct="mean")
sample_rate["sample_positive_pct"] *= 100
check = sample_rate.join(summary.set_index("appid")[["total_reviews", "positive_percentage"]])
check["gap_pct_points"] = check["sample_positive_pct"] - check["positive_percentage"]
print("Mean absolute gap (percentage points):", round(check["gap_pct_points"].abs().mean(), 1))
print("Correlation sample vs overall:", round(check[["sample_positive_pct", "positive_percentage"]].corr().iloc[0, 1], 3))
check.join(features_for_merge.set_index("appid")["name"]).sort_values("gap_pct_points").round(1)

Mean absolute gap (percentage points): 9.1
Correlation sample vs overall: 0.629


,sample_reviews,sample_positive_pct,total_reviews,positive_percentage,gap_pct_points,name
appid,,,,,,
3364770,11,36.4,1406,93.0,-56.6,Misty Judgment
2460980,7,42.9,453,76.4,-33.5,Tide—1927—
1929610,300,55.0,24475,85.1,-30.1,Demonologist
594570,300,79.3,121496,93.0,-13.6,Total War: WARHAMMER II
423620,299,52.5,775,62.6,-10.1,HERO DEFENSE
1428820,300,63.0,777,72.6,-9.6,Wordle 3
1084570,162,87.7,1116,94.7,-7.1,The Lar
412020,300,84.0,137323,89.9,-5.9,Metro Exodus
489830,300,86.7,381825,92.2,-5.5,The Elder Scrolls V: Skyrim Special Edition


In [11]:
merged = merged.drop(columns="_merge")
merged.to_csv(MERGED_OUT, index=False)
print("Saved", MERGED_OUT, merged.shape)

Saved ../data/cleaned/reviews_with_games.csv (6998, 25)


## 4. EDA – does longer playtime go with positive recommendations?

In [12]:
overall = merged["recommendation"].agg(["size", "sum", "mean"])
print(f"Reviews: {int(overall['size'])} | positive: {int(overall['sum'])} | positive rate: {overall['mean']:.1%}")

merged.groupby("recommendation")["playtime_at_review_hours"].describe(percentiles=[.25, .5, .75, .9]).rename(index={0: "Negative", 1: "Positive"}).round(1)

Reviews: 6998 | positive: 5483 | positive rate: 78.4%


,count,mean,std,min,25%,50%,75%,90%,max
recommendation,,,,,,,,,
Negative,1515.0,71.4,983.7,0.1,1.1,5.4,16.4,81.7,37059.6
Positive,5483.0,99.1,505.3,0.1,4.1,11.4,42.3,163.6,15081.2


In [13]:
def rate_table(df, by):
    t = df.groupby(by, observed=True)["recommendation"].agg(reviews="size", positive="sum")
    t["negative"] = t["reviews"] - t["positive"]
    t["positive_rate_%"] = (t["positive"] / t["reviews"] * 100).round(1)
    t["negative_rate_%"] = (100 - t["positive_rate_%"]).round(1)
    return t

by_group = rate_table(merged, "playtime_group")
by_group

,reviews,positive,negative,positive_rate_%,negative_rate_%
playtime_group,,,,,
0–2 hours,1317,834,483,63.3,36.7
2–10 hours,2267,1741,526,76.8,23.2
10–50 hours,1959,1668,291,85.1,14.9
50–100 hours,539,457,82,84.8,15.2
100+ hours,916,783,133,85.5,14.5


### Controlling for the game

Pooled numbers can be misleading: a game that is both well loved and very long (e.g. Skyrim) puts many positive reviews into the 100+ hour group, while a short game with mixed reviews fills the low groups. To check this, the positive rate is also computed **within each game** and then averaged across games (each game weighted equally).

In [14]:
per_game = merged.groupby(["appid", "playtime_group"], observed=True)["recommendation"].agg(["size", "mean"]).reset_index()
per_game = per_game[per_game["size"] >= 10]   # ignore game/group cells with fewer than 10 reviews
within = per_game.groupby("playtime_group", observed=True).agg(games=("appid", "nunique"),
                                                               mean_positive_rate_pct=("mean", lambda s: round(s.mean() * 100, 1)),
                                                               median_positive_rate_pct=("mean", lambda s: round(s.median() * 100, 1)))
within.join(by_group["positive_rate_%"].rename("pooled_positive_rate_%"))

,games,mean_positive_rate_pct,median_positive_rate_pct,pooled_positive_rate_%
playtime_group,,,,
0–2 hours,20,57.7,59.4,63.3
2–10 hours,26,78.7,83.2,76.8
10–50 hours,23,85.2,88.0,85.1
50–100 hours,14,85.5,90.3,84.8
100+ hours,12,85.5,87.2,85.5


## 5. Genres and gameplay styles

In [15]:
print("Games per primary genre in the review sample:")
review_games["primary_genre"].value_counts().to_frame("games").join(
    merged.groupby("primary_genre").size().rename("reviews"))

Games per primary genre in the review sample:


,games,reviews
primary_genre,,
Strategy,6,1731
RPG,5,948
Action,3,900
Simulation,3,900
Adventure,3,473
Horror,3,691
Roguelike,2,455
Shooter,2,600
Puzzle,1,300


In [16]:
genre_pivot = (merged.pivot_table(index="primary_genre", columns="playtime_group", values="recommendation",
                                  aggfunc="mean", observed=True) * 100).round(1)
genre_counts = merged.pivot_table(index="primary_genre", columns="playtime_group", values="recommendation",
                                  aggfunc="size", observed=True)
genre_pivot["overall_%"] = (merged.groupby("primary_genre")["recommendation"].mean() * 100).round(1)
print("Positive rate (%) by primary genre and playtime group")
genre_pivot

Positive rate (%) by primary genre and playtime group


playtime_group,0–2 hours,2–10 hours,10–50 hours,50–100 hours,100+ hours,overall_%
primary_genre,,,,,,
Action,69.8,61.1,84.9,85.7,97.4,72.4
Adventure,73.3,94.6,94.2,100.0,100.0,90.5
Horror,44.3,66.9,80.5,82.5,80.8,71.3
Puzzle,31.0,67.5,72.7,NaN,100.0,63.0
RPG,50.7,67.4,77.5,71.7,75.1,71.3
Roguelike,92.1,97.3,93.2,91.1,96.5,94.9
Shooter,59.1,85.2,86.7,97.4,85.7,83.8
Simulation,75.6,79.9,86.3,85.7,84.2,81.3
Strategy,51.0,78.7,86.5,90.3,88.2,79.6


In [17]:
print("Number of reviews behind each cell (rates from fewer than ~30 reviews are unreliable)")
genre_counts

Number of reviews behind each cell (rates from fewer than ~30 reviews are unreliable)


playtime_group,0–2 hours,2–10 hours,10–50 hours,50–100 hours,100+ hours
primary_genre,,,,,
Action,400.0,239.0,159.0,63.0,39.0
Adventure,90.0,259.0,121.0,2.0,1.0
Horror,115.0,172.0,251.0,80.0,73.0
Puzzle,42.0,234.0,22.0,NaN,2.0
RPG,67.0,285.0,253.0,106.0,237.0
Roguelike,38.0,113.0,117.0,45.0,142.0
Shooter,66.0,196.0,285.0,39.0,14.0
Simulation,254.0,234.0,226.0,91.0,95.0
Strategy,245.0,535.0,525.0,113.0,313.0


In [18]:
for col in ["gameplay_style", "story_rich", "free_to_play"]:
    print(f"\n=== {col}: games =", review_games.groupby(col)['appid'].nunique().to_dict())
    p = (merged.pivot_table(index=col, columns="playtime_group", values="recommendation",
                            aggfunc="mean", observed=True) * 100).round(1)
    p["overall_%"] = (merged.groupby(col)["recommendation"].mean() * 100).round(1)
    p["reviews"] = merged.groupby(col).size()
    display(p)


=== gameplay_style: games = {'Multiplayer': 9, 'Single-player only': 19}


playtime_group,0–2 hours,2–10 hours,10–50 hours,50–100 hours,100+ hours,overall_%,reviews
gameplay_style,,,,,,,
Multiplayer,66.0,80.3,85.2,85.9,87.5,81.4,2491
Single-player only,62.1,75.6,85.1,83.8,82.4,76.7,4507



=== story_rich: games = {False: 18, True: 10}


playtime_group,0–2 hours,2–10 hours,10–50 hours,50–100 hours,100+ hours,overall_%,reviews
story_rich,,,,,,,
False,72.2,78.7,86.7,88.2,91.7,81.4,4356
True,34.4,73.7,82.9,80.3,78.7,73.4,2642



=== free_to_play: games = {False: 26, True: 2}


playtime_group,0–2 hours,2–10 hours,10–50 hours,50–100 hours,100+ hours,overall_%,reviews
free_to_play,,,,,,,
False,58.9,75.6,84.9,84.8,85.4,77.9,6536
True,78.5,96.3,100.0,NaN,100.0,85.1,462


## 6. Long-time negative reviewers (100+ hours)

In [19]:
long = merged[merged["playtime_group"] == "100+ hours"]
neg_long = long[long["recommendation"] == 0]
print(f"100+ hour reviews: {len(long)}  | negative: {len(neg_long)}  | negative rate: {len(neg_long)/len(long):.1%}")
print(f"Share of ALL negative reviews written by 100+ hour players: {len(neg_long)/(merged['recommendation']==0).sum():.1%}")

thresholds = [100, 250, 500, 1000]
pd.DataFrame([{"min_hours": h,
               "reviews": (merged["playtime_at_review_hours"] >= h).sum(),
               "negative": ((merged["playtime_at_review_hours"] >= h) & (merged["recommendation"] == 0)).sum()}
              for h in thresholds]).assign(negative_rate_pct=lambda d: (d["negative"] / d["reviews"] * 100).round(1))

100+ hour reviews: 916  | negative: 133  | negative rate: 14.5%
Share of ALL negative reviews written by 100+ hour players: 8.8%


,min_hours,reviews,negative,negative_rate_pct
0,100,916,133,14.5
1,250,450,59,13.1
2,500,225,33,14.7
3,1000,102,8,7.8


In [20]:
print("100+ hour reviews by game (games with at least 10 such reviews)")
lg = long.groupby(["name", "primary_genre", "gameplay_style"])["recommendation"].agg(reviews_100h="size", positive="sum")
lg["negative"] = lg["reviews_100h"] - lg["positive"]
lg["negative_rate_%"] = (lg["negative"] / lg["reviews_100h"] * 100).round(1)
lg[lg["reviews_100h"] >= 10].sort_values("negative_rate_%", ascending=False)

100+ hour reviews by game (games with at least 10 such reviews)


,,,reviews_100h,positive,negative,negative_rate_%
name,primary_genre,gameplay_style,,,,
Demonologist,Horror,Multiplayer,16,9,7,43.8
Starfield,RPG,Single-player only,120,78,42,35.0
Total War: WARHAMMER II,Strategy,Multiplayer,147,116,31,21.1
WWE 2K25,Simulation,Multiplayer,69,55,14,20.3
The Elder Scrolls V: Skyrim Special Edition,RPG,Single-player only,115,98,17,14.8
Mystic Vale,Strategy,Multiplayer,15,13,2,13.3
The Outlast Trials,Horror,Multiplayer,57,50,7,12.3
Citystate,Simulation,Single-player only,24,23,1,4.2
The Binding of Isaac: Rebirth,Roguelike,Multiplayer,139,134,5,3.6


In [21]:
for col in ["primary_genre", "gameplay_style", "story_rich"]:
    t = rate_table(long, col)[["reviews", "negative", "negative_rate_%"]]
    display(t.sort_values("negative_rate_%", ascending=False))

,reviews,negative,negative_rate_%
primary_genre,,,
RPG,237,59,24.9
Horror,73,14,19.2
Simulation,95,15,15.8
Shooter,14,2,14.3
Strategy,313,37,11.8
Roguelike,142,5,3.5
Action,39,1,2.6
Puzzle,2,0,0.0
Adventure,1,0,0.0


,reviews,negative,negative_rate_%
gameplay_style,,,
Single-player only,357,63,17.6
Multiplayer,559,70,12.5


,reviews,negative,negative_rate_%
story_rich,,,
True,436,93,21.3
False,480,40,8.3


In [22]:
print("Profile of 100+ hour reviewers: negative vs positive")
long.groupby("recommendation").agg(
    reviews=("recommendationid", "size"),
    median_hours=("playtime_at_review_hours", "median"),
    mean_hours=("playtime_at_review_hours", "mean"),
    early_access_pct=("written_during_early_access", "mean"),
    received_free_pct=("received_for_free", "mean"),
    steam_purchase_pct=("steam_purchase", "mean"),
).rename(index={0: "Negative", 1: "Positive"}).round(3)

Profile of 100+ hour reviewers: negative vs positive


,reviews,median_hours,mean_hours,early_access_pct,received_free_pct,steam_purchase_pct
recommendation,,,,,,
Negative,133,224.500,698.159,0.000,0.008,0.767
Positive,783,249.833,589.978,0.006,0.010,0.801


### What do 100+ hour players complain about?

To find out *why* long-time players leave negative reviews, we read all 133 of their reviews and sorted them into complaint themes using keyword lists. A review can count toward more than one theme. We then checked the keyword matches by reading the matched reviews, and fixed lists that caught the wrong reviews (for example, WWE's "MyFaction" mode was being counted as a story complaint).

To see which complaints are *special* to long-time players, we compare them with the 1,382 negative reviews from players under 100 hours. A Fisher exact test checks each theme. Because we run 10 tests, we use a stricter cutoff of p < 0.005 (Bonferroni: 0.05 / 10).

Keyword matching is simple and can miss sarcasm or count a word used in a positive way, so these numbers are estimates.

In [23]:
import re

THEMES = {
    "Bugs, crashes or performance": r"crash|bug|broken|glitch|unplayable|corrupt|freez|fps|frame|optimi|runs like|stutter|won'?t start|wont start|game breaking|game-breaking|segmentation|close[sd]? to desktop|loading screen|stuck in",
    "Updates, patches or mods": r"update|patch|\bmods?\b|steam ?deck|anniversary|creation club|removed all",
    "Boring, repetitive or empty": r"boring|repetitive|bland|empty|soul ?less|grind|copy|same (?:buildings|science|thing)|tedious|dull|falls flat|nothing to keep|lack of|simplified",
    "Gameplay, balance or enemy AI": r"balance|difficult|luck|cheat|diplomacy|control|combat|scaling|doomstack|kick ?out|\b(?:enemy|cheating|ca'?s) ai\b|ai (?:rivals|opponents|shits)|declare war|mechanic",
    "Story, writing or quests": r"story|writing|quest|narrat|dialog|companion|characters? (?:are|talk)|constellation",
    "Developer, company or money": r"\bdlc|paid|money|price|expensive|currency|microtrans|denuvo|\bca\b|bethesda|todd|\b2k\b|\bea\b|corpo|company|greed|business|slop|deserve your",
    "Game got worse over time": r"used to|anymore|any more|worse|down ?hill|latest|recent(?:ly)?|\bnow\b|first (?:released|started)|at launch|with age|after all this time|before todd|went from",
    "Use of generative AI": r"generative ai|gen a\.?i|ai generated|use of ai|using ai|uses? (?:generative )?ai|stop ai|^ai$|use ai in|replace artists",
    "Political or social content": r"woke|\bdei\b|pronoun|political|feminist",
}
LIKED = r"i love|love (?:the|this)|great game|best game|exceptional|masterpiece|wanted to like|really tried|used to like|tried hard to have fun|huge .* fan|enjoyable"

def theme_frame(df):
    text = df["review_text"].fillna("").str.lower()
    tags = pd.DataFrame({k: text.str.contains(p, regex=True) for k, p in THEMES.items()}, index=df.index)
    tags["No real explanation"] = (text.str.split().str.len() < 4) & ~tags.any(axis=1)
    return tags

negatives = merged[merged["recommendation"] == 0]
neg_long_df = negatives[negatives["playtime_at_review_hours"] >= 100]
neg_short_df = negatives[negatives["playtime_at_review_hours"] < 100]
tl, ts = theme_frame(neg_long_df), theme_frame(neg_short_df)

rows = []
for theme in tl.columns:
    a, b = tl[theme].sum(), ts[theme].sum()
    _, p = stats.fisher_exact([[a, len(tl) - a], [b, len(ts) - b]])
    rows.append({"theme": theme, "100+ h reviews": a, "100+ h %": round(a / len(tl) * 100, 1),
                 "under 100 h %": round(b / len(ts) * 100, 1), "p_value": round(p, 4),
                 "significant (p < 0.005)": p < 0.005})
complaints = pd.DataFrame(rows).sort_values("100+ h %", ascending=False)
complaints

,theme,100+ h reviews,100+ h %,under 100 h %,p_value,significant (p < 0.005)
5,"Developer, company or money",54,40.6,21.7,0.0000,True
0,"Bugs, crashes or performance",46,34.6,25.9,0.0395,False
6,Game got worse over time,33,24.8,13.7,0.0012,True
1,"Updates, patches or mods",29,21.8,5.9,0.0000,True
2,"Boring, repetitive or empty",26,19.5,16.9,0.4689,False
3,"Gameplay, balance or enemy AI",24,18.0,16.6,0.6282,False
4,"Story, writing or quests",24,18.0,19.5,0.8181,False
9,No real explanation,10,7.5,7.2,0.8614,False
7,Use of generative AI,8,6.0,4.1,0.2681,False
8,Political or social content,4,3.0,1.2,0.1053,False


In [24]:
liked_long = neg_long_df["review_text"].fillna("").str.lower().str.contains(LIKED)
liked_short = neg_short_df["review_text"].fillna("").str.lower().str.contains(LIKED)
_, p_liked = stats.fisher_exact([[liked_long.sum(), (~liked_long).sum()], [liked_short.sum(), (~liked_short).sum()]])
print(f"Negative reviews that still say they liked/loved the game: 100+ h {liked_long.mean():.1%} vs under 100 h {liked_short.mean():.1%} (Fisher p = {p_liked:.4f})")
print(f"Share of 100+ h negative reviews from Starfield: {(neg_long_df['name'] == 'Starfield').mean():.1%}")

print("\nTop complaint themes per game (100+ h negatives, games with 10+ such reviews):")
by_game = tl.groupby(neg_long_df["name"]).mean().mul(100).round(0)
by_game = by_game[neg_long_df["name"].value_counts().reindex(by_game.index) >= 10]
by_game.T

Negative reviews that still say they liked/loved the game: 100+ h 18.0% vs under 100 h 7.5% (Fisher p = 0.0002)
Share of 100+ h negative reviews from Starfield: 31.6%

Top complaint themes per game (100+ h negatives, games with 10+ such reviews):


name,Starfield,The Elder Scrolls V: Skyrim Special Edition,Total War: WARHAMMER II,WWE 2K25
"Bugs, crashes or performance",55.0,35.0,10.0,43.0
"Updates, patches or mods",21.0,76.0,6.0,7.0
"Boring, repetitive or empty",48.0,6.0,6.0,7.0
"Gameplay, balance or enemy AI",17.0,0.0,26.0,14.0
"Story, writing or quests",50.0,0.0,6.0,0.0
"Developer, company or money",45.0,41.0,55.0,64.0
Game got worse over time,21.0,53.0,13.0,21.0
Use of generative AI,0.0,0.0,3.0,0.0
Political or social content,7.0,6.0,0.0,0.0
No real explanation,2.0,0.0,10.0,7.0


## 7. Statistical tests

All tests use α = 0.05.

### 7.1 Chi-square test of independence: playtime group × recommendation
- **H₀:** recommendation (positive/negative) is independent of playtime group.
- **H₁:** the positive rate differs between playtime groups.

Effect size is reported as **Cramér's V** (≈0.1 small, ≈0.3 medium, ≈0.5 large). Standardized residuals show which cells drive the result (|residual| > 2 is notable).

In [25]:
def chi_square_report(df, label=""):
    ct = pd.crosstab(df["playtime_group"], df["recommendation"]).rename(columns={0: "Negative", 1: "Positive"})
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    return {"subset": label, "n": n, "chi2": round(chi2, 2), "dof": dof, "p_value": p,
            "cramers_v": round(v, 3), "min_expected": round(expected.min(), 1)}, ct, expected

res, ct, expected = chi_square_report(merged, "All reviews")
print(ct, "\n")
print(pd.Series({**res, "p_value": f"{res['p_value']:.2e}"}))
resid = (ct - expected) / np.sqrt(expected)
print("\nStandardized residuals:")
resid.round(2)

recommendation  Negative  Positive
playtime_group                    
0–2 hours            483       834
2–10 hours           526      1741
10–50 hours          291      1668
50–100 hours          82       457
100+ hours           133       783 

subset          All reviews
n                      6998
chi2                 272.44
dof                       4
p_value            9.52e-58
cramers_v             0.197
min_expected          116.7
dtype: object

Standardized residuals:


recommendation,Negative,Positive
playtime_group,,
0–2 hours,11.72,-6.16
2–10 hours,1.59,-0.84
10–50 hours,-6.46,3.40
50–100 hours,-3.21,1.69
100+ hours,-4.64,2.44


### 7.2 Same test within gameplay styles and genres
Checks whether the playtime–recommendation association holds inside each subgroup. Subgroups where any expected count is below 5 are flagged because the chi-square approximation is unreliable there.

In [26]:
rows = []
for col in ["gameplay_style", "story_rich", "primary_genre"]:
    for val, sub in merged.groupby(col):
        if sub["recommendation"].nunique() < 2:
            continue
        r, _, _ = chi_square_report(sub, f"{col} = {val}")
        r["positive_rate_%"] = round(sub["recommendation"].mean() * 100, 1)
        r["reliable (min expected >= 5)"] = r["min_expected"] >= 5
        rows.append(r)
subgroup_tests = pd.DataFrame(rows)
subgroup_tests["p_value"] = subgroup_tests["p_value"].map(lambda p: f"{p:.2e}")
subgroup_tests

,subset,n,chi2,dof,p_value,cramers_v,min_expected,positive_rate_%,reliable (min expected >= 5)
0,gameplay_style = Multiplayer,2491,89.63,4,1.58e-18,0.190,48.7,81.4,True
1,gameplay_style = Single-player only,4507,173.75,4,1.64e-36,0.196,64.7,76.7,True
2,story_rich = False,4356,128.15,4,9.67e-27,0.172,56.8,81.4,True
3,story_rich = True,2642,289.04,4,2.51e-61,0.331,62.4,73.4,True
4,primary_genre = Action,900,47.02,4,1.51e-09,0.229,10.7,72.4,True
5,primary_genre = Adventure,473,38.11,4,1.06e-07,0.284,0.1,90.5,False
6,primary_genre = Horror,691,61.01,4,1.78e-12,0.297,20.9,71.3,True
7,primary_genre = Puzzle,300,22.63,3,4.83e-05,0.275,0.7,63.0,False
8,primary_genre = RPG,948,22.38,4,1.68e-04,0.154,19.2,71.3,True
9,primary_genre = Roguelike,455,4.84,4,3.04e-01,0.103,1.9,94.9,False


### 7.3 Mann–Whitney U: playtime of positive vs negative reviewers
Playtime is heavily right-skewed, so a rank-based test is used instead of a t-test.
- **H₀:** playtime distributions are the same for positive and negative reviewers.

In [27]:
pos = merged.loc[merged["recommendation"] == 1, "playtime_at_review_hours"]
neg = merged.loc[merged["recommendation"] == 0, "playtime_at_review_hours"]
u, p = stats.mannwhitneyu(pos, neg, alternative="two-sided")
rank_biserial = 1 - 2 * u / (len(pos) * len(neg))   # sign: negative => positive group tends higher
print(f"median hours – positive: {pos.median():.1f}, negative: {neg.median():.1f}")
print(f"U = {u:.0f}, p = {p:.2e}, rank-biserial r = {-rank_biserial:.3f} (positive => positive reviewers played longer)")

median hours – positive: 11.4, negative: 5.4
U = 5251440, p = 4.58e-56, rank-biserial r = 0.264 (positive => positive reviewers played longer)


### 7.4 Within-game test (controls for which game was reviewed)
Reviews are clustered within 28 games, so the pooled tests above partly compare *games* rather than *players*. Here each game is one observation: for every game with at least 10 reviews both below and above its own median playtime, we compare the positive rate of its longer-playing half against its shorter-playing half using a **Wilcoxon signed-rank test**.
- **H₀:** within a game, longer- and shorter-playing reviewers recommend at the same rate.

In [28]:
rows = []
for appid, g in merged.groupby("appid"):
    med = g["playtime_at_review_hours"].median()
    lo, hi = g[g["playtime_at_review_hours"] <= med], g[g["playtime_at_review_hours"] > med]
    if len(lo) >= 10 and len(hi) >= 10:
        rows.append({"name": g["name"].iat[0], "gameplay_style": g["gameplay_style"].iat[0],
                     "median_hours": round(med, 1), "short_half_pos_%": round(lo["recommendation"].mean() * 100, 1),
                     "long_half_pos_%": round(hi["recommendation"].mean() * 100, 1)})
paired = pd.DataFrame(rows)
paired["difference"] = paired["long_half_pos_%"] - paired["short_half_pos_%"]
nonzero = paired.loc[paired["difference"] != 0, "difference"]
w, p = stats.wilcoxon(nonzero)
print(f"Games compared: {len(paired)} | long half higher in {(paired['difference'] > 0).sum()}, lower in {(paired['difference'] < 0).sum()}, tied in {(paired['difference'] == 0).sum()}")
print(f"Median difference: {paired['difference'].median():.1f} pts | Wilcoxon W = {w:.1f}, p = {p:.2e}")
paired.sort_values("difference", ascending=False)

Games compared: 26 | long half higher in 23, lower in 2, tied in 1
Median difference: 10.7 pts | Wilcoxon W = 7.0, p = 2.86e-05


,name,gameplay_style,median_hours,short_half_pos_%,long_half_pos_%,difference
6,HERO DEFENSE,Single-player only,5.8,34.7,70.5,35.8
16,Disc Space,Multiplayer,0.8,70.0,97.3,27.3
10,Devil's Hunt,Single-player only,4.5,26.7,53.3,26.6
3,Oh My Gore!,Single-player only,6.3,72.4,92.2,19.8
13,The Lar,Single-player only,2.4,77.8,97.5,19.7
0,Sid Meier's Civilization® III Complete,Multiplayer,41.2,77.3,94.7,17.4
1,Command & Conquer™ Red Alert™ 3,Single-player only,14.1,80.0,96.7,16.7
9,Citystate,Single-player only,12.9,72.8,89.3,16.5
23,Vampire: The Masquerade - Reckoning of New York,Single-player only,7.8,56.5,71.8,15.3
25,EGG RAIDERS,Multiplayer,0.4,48.9,63.6,14.7


### 7.5 Are 100+ hour players less likely to leave negative reviews than everyone else?
Two-proportion comparison (2×2 chi-square) of the negative rate for 100+ hour reviewers vs. all others, plus the same comparison of long-time negative rates between gameplay styles.

In [29]:
merged["is_100h"] = merged["playtime_group"] == "100+ hours"
ct2 = pd.crosstab(merged["is_100h"], merged["recommendation"]).rename(index={False: "< 100 h", True: "100+ h"}, columns={0: "Negative", 1: "Positive"})
chi2, p, _, _ = stats.chi2_contingency(ct2)
print(ct2.assign(negative_rate_pct=lambda d: (d["Negative"] / d.sum(axis=1) * 100).round(1)))
print(f"chi2 = {chi2:.2f}, p = {p:.2e}\n")

ct3 = pd.crosstab(long["gameplay_style"], long["recommendation"]).rename(columns={0: "Negative", 1: "Positive"})
chi2, p, _, exp3 = stats.chi2_contingency(ct3)
print(ct3.assign(negative_rate_pct=lambda d: (d["Negative"] / d.sum(axis=1) * 100).round(1)))
print(f"100+ h reviewers, multiplayer vs single-player only: chi2 = {chi2:.2f}, p = {p:.3g}, min expected = {exp3.min():.1f}")

recommendation  Negative  Positive  negative_rate_pct
is_100h                                              
< 100 h             1382      4700               22.7
100+ h               133       783               14.5
chi2 = 31.10, p = 2.45e-08

recommendation      Negative  Positive  negative_rate_pct
gameplay_style                                           
Multiplayer               70       489               12.5
Single-player only        63       294               17.6
100+ h reviewers, multiplayer vs single-player only: chi2 = 4.21, p = 0.0403, min expected = 51.8


## 8. Summary of findings

The cell below prints the key numbers so this section stays in sync with the data. Interpretation notes:

- **Limitations.** 28 games is a small game sample; genre groups contain only 1–7 games each, so genre comparisons describe *these games* more than genres in general. Reviews come from Steam's most recent reviews, so they lean toward current players. Recommendation is a proxy for satisfaction, not a direct measure of it.
- **Pooled vs within-game.** Pooled tests treat ~7,000 reviews as independent even though they cluster in 28 games; the within-game test (7.4) is the more conservative evidence.

In [30]:
res_all, _, _ = chi_square_report(merged, "All")
lines = [
    f"Merged dataset: {len(merged)} reviews across {merged['appid'].nunique()} games; no unmatched or duplicated rows.",
    f"Overall positive rate: {merged['recommendation'].mean():.1%}.",
    "Positive rate by playtime group: " + ", ".join(f"{g} {r:.1f}%" for g, r in by_group['positive_rate_%'].items()) + ".",
    f"Chi-square (playtime group × recommendation): chi2 = {res_all['chi2']}, dof = {res_all['dof']}, p = {res_all['p_value']:.2e}, Cramér's V = {res_all['cramers_v']}.",
    f"Median playtime: positive reviewers {pos.median():.1f} h vs negative reviewers {neg.median():.1f} h (Mann–Whitney p = {stats.mannwhitneyu(pos, neg).pvalue:.2e}).",
    f"Within-game: longer-playing half had a higher positive rate in {(paired['difference'] > 0).sum()} of {len(paired)} games (median +{paired['difference'].median():.1f} pts, Wilcoxon p = {stats.wilcoxon(nonzero).pvalue:.2e}).",
    f"100+ hour reviewers: {len(long)} reviews, {len(neg_long)} negative ({len(neg_long)/len(long):.1%}); vs {ct2.loc['< 100 h','Negative']/ct2.loc['< 100 h'].sum():.1%} negative among players under 100 h.",
]
print("\n".join("• " + l for l in lines))

• Merged dataset: 6998 reviews across 28 games; no unmatched or duplicated rows.
• Overall positive rate: 78.4%.
• Positive rate by playtime group: 0–2 hours 63.3%, 2–10 hours 76.8%, 10–50 hours 85.1%, 50–100 hours 84.8%, 100+ hours 85.5%.
• Chi-square (playtime group × recommendation): chi2 = 272.44, dof = 4, p = 9.52e-58, Cramér's V = 0.197.
• Median playtime: positive reviewers 11.4 h vs negative reviewers 5.4 h (Mann–Whitney p = 4.58e-56).
• Within-game: longer-playing half had a higher positive rate in 23 of 26 games (median +10.7 pts, Wilcoxon p = 2.86e-05).
• 100+ hour reviewers: 916 reviews, 133 negative (14.5%); vs 22.7% negative among players under 100 h.


### Interpretation (based on the November 2026 sample)

1. **Longer playtime is associated with more positive recommendations, but only up to a point.** The positive rate climbs from ~63% (0–2 h) to ~77% (2–10 h) to ~85% (10–50 h), then flattens: 50–100 h and 100+ h players are no more positive than 10–50 h players. The chi-square test is highly significant with a small-to-moderate effect (Cramér's V ≈ 0.20), and the residuals show the effect is driven mostly by the many negative reviews from players under 2 hours.
2. **It is not just a "which game" effect.** Within individual games, the longer-playing half of reviewers recommended the game more often in 23 of 26 games (median +10.7 points, Wilcoxon p < 0.001).
3. **The pattern holds across gameplay styles.** Multiplayer and single-player-only games show nearly the same curve and effect size. Story-rich games show the *strongest* association (V ≈ 0.33): very short sessions (0–2 h) are only ~34% positive, likely players who quit early on a narrative game.
4. **Genres differ in level more than in shape.** Most genres follow the same rising curve; Roguelike games are positive at every playtime level, so there is no significant association there. Several genre cells are too small for a reliable chi-square (flagged in 7.2).
5. **Long-time negative reviewers are a real but minority group.** 14.5% of 100+ hour reviews are negative (vs 22.7% under 100 h). They concentrate in a few games: Starfield (35%), Demonologist (44%, small n), Total War: WARHAMMER II (21%) and WWE 2K25 (20%). Single-player-only and story-rich games have higher long-time negative rates (17.6% and 21.3%) than multiplayer (12.5%) and non-story games (8.3%), which is consistent with players finishing a long game and then judging it, whereas multiplayer players who dislike a game tend to stop playing earlier.
6. **What long-time critics complain about is different.** Compared with shorter-playtime negative reviews, 100+ hour critics much more often mention updates or mods breaking the game, the game getting worse over time, and the developer/company. Many of them say they used to love the game. Their problem is usually not that the game was never fun; it is that something changed.

**Caveats:** 28 games, small genre groups, recent-review sampling (several small games' sample rates differ from their all-time Steam rate), and recommendations as a proxy for satisfaction. These are associations, not causal effects.